<a href="https://colab.research.google.com/github/rcopty/asp360-movie-flop-predictor/blob/main/Untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Environment Setup

In [ ]:
%pip install -q "kagglehub[pandas-datasets]"

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import json
import pandas as pd
import numpy as np

## 2. Load the TMDB Datasets

In [ ]:
# Load movies file
df_movies = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "tmdb/tmdb-movie-metadata",
  "tmdb_5000_movies.csv"
)

# Load credits file
df_credits = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "tmdb/tmdb-movie-metadata",
  "tmdb_5000_credits.csv"
)

print(df_movies.shape)
print(df_credits.shape)
print(df_movies.columns.tolist())
print(df_credits.columns.tolist())

## Dataset Inspection

In [ ]:
print(df_movies.shape)
print(df_credits.shape)
display(df_movies.head())
display(df_credits.head())
df_movies.info()

## Example Movies

In [ ]:
examples = ['John Carter', 'The Lone Ranger', 'Green Lantern']

for title in examples:
    row = df_movies[df_movies['title'] == title]
    if len(row) > 0:
        budget = row['budget'].values[0]
        revenue = row['revenue'].values[0]
        print(f"{title}: budget=${budget/1e6:.0f}M, revenue=${revenue/1e6:.0f}M, ratio={revenue/budget:.2f}")

## Usable Movies

In [ ]:
usable = df_movies[
    (df_movies['budget'] > 50_000_000) &
    (df_movies['revenue'] > 0)
]
print(f"Usable movies (budget > $50M and revenue > 0): {len(usable)}")

## Cast and Crew Info

In [ ]:
# Extract director
def get_director(crew_json):
    try:
        crew = json.loads(crew_json)
        for member in crew:
            if member.get('job') == 'Director':
                return member.get('name')
    except:
        return None
    return None

# Extract lead actor = first listed cast member
def get_lead_actor(cast_json):
    try:
        cast = json.loads(cast_json)
        if len(cast) > 0:
            return cast[0].get('name')
    except:
        return None
    return None

# Merge movies and credits
df_all = df_movies.merge(df_credits[['movie_id', 'crew', 'cast']], left_on='id', right_on='movie_id')

df_all['director'] = df_all['crew'].apply(get_director)
df_all['lead_actor'] = df_all['cast'].apply(get_lead_actor)

# valid budget/revenue mask
valid_financials = (df_all['budget'] > 0) & (df_all['revenue'] > 0)

# Flop label
df_all['flop'] = np.nan
df_all.loc[valid_financials, 'flop'] = (
    df_all.loc[valid_financials, 'revenue'] < 2.5 * df_all.loc[valid_financials, 'budget']).astype(int)

# Revenue-to-budget ratio
df_all['rev_budget_ratio'] = np.nan
df_all.loc[valid_financials, 'rev_budget_ratio'] = (
    df_all.loc[valid_financials, 'revenue'] / df_all.loc[valid_financials, 'budget'])

# compute prior track records & number of prior valid films
df_all['release_date'] = pd.to_datetime(df_all['release_date'], errors='coerce')
df_all = df_all.sort_values('release_date').reset_index(drop=True)
df_all['rev_budget_ratio_capped'] = df_all['rev_budget_ratio'].clip(upper=10) # cap so blockbuster does not dominate
df_all['director_success_ratio'] = df_all.groupby('director')['rev_budget_ratio_capped'].transform(
    lambda x: x.shift(1).expanding().mean())
df_all['actor_success_ratio'] = df_all.groupby('lead_actor')['rev_budget_ratio_capped'].transform(
    lambda x: x.shift(1).expanding().mean())
df_all['director_prior_count'] = df_all.groupby('director')['rev_budget_ratio_capped'].transform(
    lambda x: x.shift(1).expanding().count())
df_all['actor_prior_count'] = df_all.groupby('lead_actor')['rev_budget_ratio_capped'].transform(
    lambda x: x.shift(1).expanding().count())

# Filter to mid-to-high-budget films
df = df_all[
    (df_all['budget'] > 50_000_000) &
    (df_all['revenue'] > 0)].copy()

# Fill missing track records
# 1.0 means revenue equals budget, a neutral/default commercial performance value.
df['director_success_ratio'] = df['director_success_ratio'].fillna(1.0)
df['actor_success_ratio'] = df['actor_success_ratio'].fillna(1.0)
# 0 means no prior valid films in the dataset
df['director_prior_count'] = df['director_prior_count'].fillna(0)
df['actor_prior_count'] = df['actor_prior_count'].fillna(0)

# release year for chronological splitting
df['release_year'] = df['release_date'].dt.year

print(f"Final dataset shape: {df.shape}")

print(df[
    ['title', 'release_year', 'director', 'lead_actor',
     'director_success_ratio', 'director_prior_count',
     'actor_success_ratio', 'actor_prior_count',
     'flop']
].head(10).to_string(index=False))

## Data Distribution

In [ ]:
print("FLOP vs HIT")
print(f"Total movies: {len(df)}")
print(f"Flops: {int(df['flop'].sum())} ({df['flop'].mean()*100:.1f}%)")
print(f"Hits: {int((df['flop'] == 0).sum())} ({(1 - df['flop'].mean())*100:.1f}%)")

print("\nDIRECTOR TRACK RECORD")
has_real_director = df['director_prior_count'] > 0
print(f"Films with real director track record: {has_real_director.sum()} ({has_real_director.mean()*100:.1f}%)")
print(f"Films with no prior director data: {(~has_real_director).sum()}")

print("\nACTOR TRACK RECORD")
has_real_actor = df['actor_prior_count'] > 0
print(f"Films with real actor track record: {has_real_actor.sum()} ({has_real_actor.mean()*100:.1f}%)")
print(f"Films with no prior actor data: {(~has_real_actor).sum()}")

print("\nSAMPLE ROWS")
print(df[['title', 'release_year', 'director', 'lead_actor',
     'director_success_ratio', 'director_prior_count',
     'actor_success_ratio', 'actor_prior_count',
     'flop']].head(15).to_string(index=False))

In [ ]:
# Yearly distribution after final filtering
df['release_year'] = df['release_date'].dt.year

yearly_dist = (
    df.groupby('release_year')
      .agg(
          total_movies=('title', 'count'),
          flops=('flop', 'sum'),
          hits=('flop', lambda x: (x == 0).sum()),
          flop_rate=('flop', 'mean')
      )
      .reset_index()
)

yearly_dist['flop_rate'] = (yearly_dist['flop_rate'] * 100).round(1)

print("YEARLY DISTRIBUTION")
print(yearly_dist.to_string(index=False))

In [ ]:
train = df[df['release_year'] < 2010]
val = df[(df['release_year'] >= 2010) & (df['release_year'] <= 2013)]
test = df[df['release_year'] >= 2014]

print("SPLIT DISTRIBUTION")
for name, split in [('Train (<2010)', train),
                    ('Validation (2010-2013)', val),
                    ('Test (2014+)', test)]:
    print(f"\n{name}")
    print(f"Movies: {len(split)}")
    print(f"Flops: {split['flop'].sum()} ({split['flop'].mean()*100:.1f}%)")
    print(f"Hits: {(split['flop']==0).sum()} ({(1-split['flop'].mean())*100:.1f}%)")
    print(f"Years: {split['release_year'].min()} - {split['release_year'].max()}")